In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
%pip install -U dagshub mlflow skops --quiet

Note: you may need to restart the kernel to use updated packages.


In [3]:
import dagshub
dagshub.init(repo_owner='gvakh23', repo_name='ML_assignment2', mlflow=True)

Accessing as gvakh23

Initialized MLflow to track repo "gvakh23/ML_assignment2"

Repository gvakh23/ML_assignment2 initialized!

In [4]:
import mlflow
print(mlflow.get_tracking_uri())

https://dagshub.com/gvakh23/ML_assignment2.mlflow


In [5]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

train_trans  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_ident  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

train = train_trans.merge(train_ident, on='TransactionID', how='left')


In [6]:
train_sorted = train.sort_values('TransactionDT')

val_size = int(len(train_sorted) * 0.2)
train_df = train_sorted.iloc[:-val_size].copy()
val_df   = train_sorted.iloc[-val_size:].copy()

print(f"Train split : {train_df.shape}, fraud rate: {train_df['isFraud'].mean():.2%}")
print(f"Val   split : {val_df.shape},   fraud rate: {val_df['isFraud'].mean():.2%}")


Train split : (472432, 434), fraud rate: 3.51%
Val   split : (118108, 434),   fraud rate: 3.44%


## Cleaning

In [7]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold
class Cleaner(BaseEstimator, TransformerMixin):
    """
    Drops high-null columns, encodes M columns (T/F→1/0),
    fills remaining NaN with -999.
    All thresholds computed on train only.
    """
 
    def __init__(self, null_threshold=0.9, fill_value=-999):
        self.null_threshold = null_threshold
        self.fill_value     = fill_value
 
    def fit(self, X, y=None):
        print("clean_in")

        df = X.copy()
 
        null_rates = df.isnull().mean()
        self.cols_to_drop_ = null_rates[null_rates > self.null_threshold].index.tolist()
        self.cols_to_drop_ = [c for c in self.cols_to_drop_
                               if c not in ['isFraud', 'TransactionID']]
 
        # Learn M column names
        self.m_cols_ = [c for c in df.columns if c.startswith('M')]
 
        with mlflow.start_run(run_name='RandomForest_Cleaning', nested=True):
            mlflow.log_param('null_threshold',  self.null_threshold)
            mlflow.log_param('fill_value',      self.fill_value)
            mlflow.log_param('fill_reason',     'tree_model_out_of_range_sentinel')
            mlflow.log_param('m_col_encoding',  'T=1_F=0_NaN=fill_value')
            mlflow.log_metric('cols_dropped',   len(self.cols_to_drop_))
            mlflow.log_metric('cols_remaining', df.shape[1] - len(self.cols_to_drop_))
            mlflow.log_metric('rows',           df.shape[0])
            mlflow.log_metric('fraud_rate',     float(y.mean()) if y is not None else -1)
        print(f"[Cleaner] Done — dropped {len(self.cols_to_drop_)} cols, {df.shape[0]} rows")
        return self
 
    def transform(self, X):
        print("clean trans in")
        df = X.copy()
 
        # Drop high-null cols
        df.drop(columns=[c for c in self.cols_to_drop_ if c in df.columns],
                inplace=True, errors='ignore')
 
        # M columns T/F → 1/0
        for col in self.m_cols_:
            if col in df.columns:
                df[col] = df[col].map({'T': 1, 'F': 0})
 
        # Fill NaN
        df.fillna(self.fill_value, inplace=True)
 
        return df

## Feature Engineering

In [8]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.d_cols      = ['D1', 'D2', 'D3', 'D4', 'D5', 'D10', 'D15']
        self.freq_cols   = ['card1', 'card2', 'addr1', 'uid', 'uid2', 'P_emaildomain']
        self.agg_cols    = ['card1', 'uid']          # removed card2/addr1 — less useful
        self.agg_targets = ['TransactionAmt', 'D1', 'C1', 'C2', 'C13']

    def fit(self, X, y=None):
        df = X.copy()

        # 1. D normalization mins
        self.d_min_map_ = {}
        for d in self.d_cols:
            if d in df.columns:
                self.d_min_map_[d] = df.groupby('card1')[d].min()

        # 2. Build UIDs
        df['uid']  = df['card1'].astype(str) + '_' + df['card2'].astype(str)
        # uid2: use D1 only if it's not -999 (missing)
        d1_str     = df['D1'].apply(lambda x: str(int(x)) if x != -999 else 'NA')
        df['uid2'] = df['card1'].astype(str) + '_' + df['addr1'].astype(str) + '_' + d1_str

        # 3. Frequency maps
        self.freq_maps_ = {}
        for col in self.freq_cols:
            if col in df.columns:
                self.freq_maps_[col] = df[col].value_counts()

        # 4. Aggregation maps — store as dict of Series for fast map()
        self.agg_maps_ = {}
        for col in self.agg_cols:
            if col not in df.columns:
                continue
            self.agg_maps_[col] = {}
            for target in self.agg_targets:
                if target not in df.columns:
                    continue
                self.agg_maps_[col][target] = {}
                grp = df.groupby(col)[target]
                self.agg_maps_[col][target]['mean'] = grp.mean()
                self.agg_maps_[col][target]['std']  = grp.std()
                self.agg_maps_[col][target]['max']  = grp.max()

        # 5. UID label encoders — store as plain dict for fastest mapping
        self.uid_maps_ = {}
        for col in ['uid', 'uid2']:
            if col in df.columns:
                unique_vals = df[col].astype(str).unique()
                self.uid_maps_[col] = {v: i for i, v in enumerate(unique_vals)}

        # 6. Email suffix encoder
        if 'P_emaildomain' in df.columns:
            suffixes = df['P_emaildomain'].astype(str).str.split('.').str[-1]
            unique_sfx = suffixes.unique()
            self.email_suffix_map_ = {v: i for i, v in enumerate(unique_sfx)}
        else:
            self.email_suffix_map_ = {}


        with mlflow.start_run(run_name='RF_Feature_Engineering', nested=True):
            mlflow.log_param('time_features',      'hour, day')
            mlflow.log_param('log_transform',      'TransactionAmt_log, TransactionAmt_decimal')
            mlflow.log_param('d_cols_normalized',  str([d for d in self.d_cols if d in self.d_min_map_]))
            mlflow.log_param('d_norm_method',      'subtract_card1_group_min_train_only')
            mlflow.log_param('uid',                'card1+card2')
            mlflow.log_param('uid2',               'card1+addr1+D1 (NA if D1 missing)')
            mlflow.log_param('freq_cols',          str(self.freq_cols))
            mlflow.log_param('agg_cols',           str(self.agg_cols))
            mlflow.log_param('agg_targets',        str(self.agg_targets))
            mlflow.log_param('email_features',     'same_email, P_email_suffix')
            mlflow.log_metric('d_cols_fitted',     len(self.d_min_map_))
            mlflow.log_metric('freq_cols_fitted',  len(self.freq_maps_))
            mlflow.log_metric('uid_maps_fitted',   len(self.uid_maps_))

        return self

    def transform(self, X):
        df = X.copy()

        # ── Time features ─────────────────────────────────────
        df['hour']  = (df['TransactionDT'] / 3600) % 24
        df['day']   = (df['TransactionDT'] / (3600 * 24)) % 7

        # ── Log transform ─────────────────────────────────────
        df['TransactionAmt_log']     = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_decimal'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)

        # ── Email features ────────────────────────────────────
        if 'P_emaildomain' in df.columns and 'R_emaildomain' in df.columns:
            df['same_email'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
            # Encode suffix using saved map (fast dict lookup, no lambda)
            suffixes = df['P_emaildomain'].astype(str).str.split('.').str[-1]
            df['P_email_suffix'] = suffixes.map(self.email_suffix_map_).fillna(-1).astype(int)

        # ── D normalization ───────────────────────────────────
        for d, min_map in self.d_min_map_.items():
            if d in df.columns:
                df[f'{d}_norm'] = df[d] - df['card1'].map(min_map).fillna(0)

        # ── UIDs ──────────────────────────────────────────────
        df['uid']  = df['card1'].astype(str) + '_' + df['card2'].astype(str)
        d1_str     = df['D1'].apply(lambda x: str(int(x)) if x != -999 else 'NA')
        df['uid2'] = df['card1'].astype(str) + '_' + df['addr1'].astype(str) + '_' + d1_str

        # ── Frequency encoding ────────────────────────────────
        for col, freq_map in self.freq_maps_.items():
            if col in df.columns:
                df[f'{col}_freq'] = df[col].map(freq_map).fillna(0)

        # ── Aggregations ──────────
        for col, targets in self.agg_maps_.items():
            if col not in df.columns:
                continue
            for target, stats in targets.items():
                for stat, series in stats.items():
                    df[f'{col}_{target}_{stat}'] = df[col].map(series).fillna(-999)

        # ── UID encoding ──────────────
        for col, uid_map in self.uid_maps_.items():
            if col in df.columns:
                df[col] = df[col].astype(str).map(uid_map).fillna(-1).astype(int)

        return df




## Categorical Encoding

In [9]:
import pandas as pd
import mlflow
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder

class CategoricalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, ohe_threshold=10):
        self.ohe_threshold = ohe_threshold
        self.device_map_ = {'desktop': 1, 'mobile': 0}
        self.ohe_info_ = {}
        self.le_encoders_ = {}
        self.le_cols_ = []

    def fit(self, X, y=None):
        print("encode_in")

        df = X.copy()
        
        # 1. Identify Categorical Columns (excluding IDs/Targets)
        all_cat_cols = [c for c in df.select_dtypes(include='object').columns 
                        if c not in ['TransactionID', 'isFraud', 'DeviceType']]

        # 2. Split logic: OHE vs Label Encode based on threshold
        for col in all_cat_cols:
            n_unique = df[col].nunique()
            if n_unique < self.ohe_threshold:
                self.ohe_info_[col] = df[col].unique().tolist()
            else:
                le = LabelEncoder()
                le.fit(df[col].astype(str))
                self.le_encoders_[col] = le
                self.le_cols_.append(col)

        # MLflow logging (matches your previous class style)
        with mlflow.start_run(run_name='RandomForest_Categorical_Encoding', nested=True):
            mlflow.log_param('ohe_threshold', self.ohe_threshold)
            mlflow.log_param('ohe_cols', list(self.ohe_info_.keys()))
            mlflow.log_param('le_cols', self.le_cols_)
            mlflow.log_metric('ohe_count', len(self.ohe_info_))
            mlflow.log_metric('le_count', len(self.le_cols_))
        print(f"[CategoricalEncoder] Done — OHE cols: {len(self.ohe_info_)}, LE cols: {len(self.le_cols_)}")

        return self

    def transform(self, X):
        print("encode trans in")

        df = X.copy()

        # ── BINARY: DeviceType ──
        if 'DeviceType' in df.columns:
            df['DeviceType'] = df['DeviceType'].map(self.device_map_).fillna(-999)

        # ── ONE-HOT ENCODING ──
        for col, categories in self.ohe_info_.items():
            if col in df.columns:
        
                dummies = pd.get_dummies(df[col], prefix=col)
        
                expected_cols = [f"{col}_{cat}" for cat in categories]
                dummies = dummies.reindex(columns=expected_cols, fill_value=0)
        
                df = pd.concat([df, dummies], axis=1)
                df.drop(columns=[col], inplace=True)
                
        # ── LABEL ENCODING ──
        for col, le in self.le_encoders_.items():
            if col in df.columns:
                # Handle unseen categories by mapping to -1
                known_classes = set(le.classes_)
                mapping = {cls: idx for idx, cls in enumerate(le.classes_)}
                df[col] = df[col].astype(str).map(mapping).fillna(-1)

        return df

## Feature Selection

In [10]:
import pandas as pd
import numpy as np
import mlflow
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold

class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, variance_threshold=0.01):
        self.variance_threshold = variance_threshold
        self.drop_cols_ = []
        self.feature_cols_ = []

    def fit(self, X, y=None):
        print("select_in")

        df = X.copy()
        numeric_df = df.select_dtypes(include=[np.number])
        
        # ──  Variance Threshold ──
        vt = VarianceThreshold(threshold=self.variance_threshold)
        vt.fit(numeric_df)
        low_var_cols = numeric_df.columns[~vt.get_support()].tolist()

        # ──  V-Column NaN-Group Selection ──
        # In this dataset, V columns are often redundant. 
        # Group them by their 'NaN pattern' and keep only the one strongest with target.
        v_cols = [c for c in df.columns if c.startswith('V')]
        
        null_patterns = {}
        for col in v_cols:
            pattern = int((df[col] == -999).sum())
            null_patterns.setdefault(pattern, []).append(col)

        best_per_group = []
        if y is not None:
            # Convert y to series for easy correlation mapping
            y_s = pd.Series(y.values if hasattr(y, 'values') else np.array(y), index=df.index)
            for pattern, cols in null_patterns.items():
                corrs = df[cols].corrwith(y_s).abs().fillna(0)
                best_per_group.append(corrs.idxmax())
        else:
            for pattern, cols in null_patterns.items():
                best_per_group.append(cols[0])

        v_to_drop = [c for c in v_cols if c not in best_per_group]
        
        # Combine all columns to remove
        self.drop_cols_ = list(set(low_var_cols + v_to_drop))
        
        # Save the "survivor" column names in order
        self.feature_cols_ = [c for c in df.columns if c not in self.drop_cols_]

        # MLflow Logging
        with mlflow.start_run(run_name='RandomForest_Feature_Selection', nested=True):
            mlflow.log_param('variance_threshold', self.variance_threshold)
            mlflow.log_metric('v_nan_groups', len(null_patterns))
            mlflow.log_metric('total_features_dropped', len(self.drop_cols_))
            mlflow.log_metric('final_feature_count', len(self.feature_cols_))
        print(f"[FeatureSelector] Done — dropped {len(self.drop_cols_)} features, {len(self.feature_cols_)} remaining")

        return self

    def transform(self, X):
        print("select trans in")

        df = X.copy()
        
        # Drop the identified useless columns
        df.drop(columns=[c for c in self.drop_cols_ if c in df.columns], 
                inplace=True, errors='ignore')
        
        df = df.reindex(columns=self.feature_cols_, fill_value=0)
        
        return df

### Preprocessing Pipeline

In [11]:
from sklearn.pipeline import Pipeline

NULL_THRESH = 0.8
OHE_THRESH  = 10
VAR_THRESH  = 0.01


import mlflow
import mlflow.sklearn

mlflow.set_experiment("RandomForest_Training")

preprocessing = Pipeline([
    ('cleaning',     Cleaner(null_threshold=NULL_THRESH)),
    ('feature_eng',  FeatureEngineer()),
    ('encoding',     CategoricalEncoder(ohe_threshold=OHE_THRESH)),
    ('selection',    FeatureSelector(variance_threshold=VAR_THRESH)),
])



In [12]:
X_train = train_df.drop(columns=['isFraud', 'TransactionID'])
y_train = train_df['isFraud']

X_val = val_df.drop(columns=['isFraud', 'TransactionID'])
y_val = val_df['isFraud']


In [15]:
X_train = preprocessing.fit_transform(X_train, y_train)
X_val   = preprocessing.transform(X_val)

print("Preprocessing Done")

clean_in
🏃 View run RandomForest_Cleaning at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2/runs/802819bf5a5f4fbcb8ace150f3863448
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2
[Cleaner] Done — dropped 74 cols, 472432 rows
clean trans in
🏃 View run RF_Feature_Engineering at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2/runs/af27e76ba5ee4a7fbcebfd8817178c2c
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2
encode_in
🏃 View run RandomForest_Categorical_Encoding at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2/runs/3054155fb90344c98b0cf69e828ecd16
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2
[CategoricalEncoder] Done — OHE cols: 12, LE cols: 4
encode trans in
select_in
🏃 View run RandomForest_Feature_Selection at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2/runs/2d7afa061d534209ba7d90fb

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline

## Training

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline


with mlflow.start_run(run_name='RF_Underfitted'):
    # Very shallow trees (max_depth=3) cannot capture complex fraud patterns
    # Few estimators (10) → high variance, low model capacity
    params = {'n_estimators': 10, 'max_depth': 3,
              'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1}
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(val_df['isFraud'], model.predict_proba(X_val)[:, 1])
    mlflow.log_params(params)
    mlflow.log_metric('train_auc',   train_auc)
    mlflow.log_metric('val_auc',     val_auc)
    mlflow.log_metric('overfit_gap', train_auc - val_auc)
    print(f'UNDERFITTED — train: {train_auc:.4f}, val: {val_auc:.4f}, gap: {train_auc-val_auc:.4f}')


UNDERFITTED — train: 0.8148, val: 0.8093, gap: 0.0055
🏃 View run RF_Underfitted at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2/runs/058d3c549ba742579c3291e03917fc9a
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2


### Undersampling

In [25]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(sampling_strategy=0.1, random_state=42)
X_train_us, y_train_us = rus.fit_resample(X_train, y_train)

print(f"Before — fraud: {y_train.sum():,}, non-fraud: {(y_train==0).sum():,}")
print(f"After  — fraud: {y_train_us.sum():,}, non-fraud: {(y_train_us==0).sum():,}")
print(f"New shape: {X_train_us.shape}")

Before — fraud: 16,599, non-fraud: 455,833
After  — fraud: 16,599, non-fraud: 165,990
New shape: (182589, 180)


In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline


with mlflow.start_run(run_name='RF_Underfitted_Undersampled'):
    params = {'n_estimators': 10, 'max_depth': 3,
              'class_weight': None, 'random_state': 42, 'n_jobs': -1}
    model = RandomForestClassifier(**params)
    model.fit(X_train_us, y_train_us)
    train_auc = roc_auc_score(y_train_us, model.predict_proba(X_train_us)[:, 1])
    val_auc   = roc_auc_score(val_df['isFraud'], model.predict_proba(X_val)[:, 1])
    mlflow.log_params(params)
    mlflow.log_metric('train_auc',   train_auc)
    mlflow.log_metric('val_auc',     val_auc)
    mlflow.log_metric('overfit_gap', train_auc - val_auc)
    print(f'UNDERFITTED — train: {train_auc:.4f}, val: {val_auc:.4f}, gap: {train_auc-val_auc:.4f}')


UNDERFITTED — train: 0.8293, val: 0.8260, gap: 0.0033
🏃 View run RF_Underfitted_Undersampled at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2/runs/317ff384dc274879b590286055f60879
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2


In [20]:
with mlflow.start_run(run_name='RF_Overfitted'):
    params = {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 1,
              'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1}
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(val_df['isFraud'], model.predict_proba(X_val)[:, 1])
    mlflow.log_params(params)
    mlflow.log_metric('train_auc',   train_auc)
    mlflow.log_metric('val_auc',     val_auc)
    mlflow.log_metric('overfit_gap', train_auc - val_auc)
    print(f'OVERFITTED  — train: {train_auc:.4f}, val: {val_auc:.4f}, gap: {train_auc-val_auc:.4f}')


OVERFITTED  — train: 1.0000, val: 0.9043, gap: 0.0957
🏃 View run RF_Overfitted at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2/runs/88020985274e4b3fa01c932d1cb8ca14
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2


In [24]:
with mlflow.start_run(run_name='RF_Baseline_Undersampled') as run:
    params = {'n_estimators': 200, 'max_depth': 15, 'min_samples_leaf': 10,
              'max_features': 'sqrt', 'class_weight': None,
              'random_state': 42, 'n_jobs': -1}
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    for fold, (tr_idx, cv_idx) in enumerate(tscv.split(X_train)):
        m = RandomForestClassifier(**params)
        m.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        score = roc_auc_score(y_train.iloc[cv_idx], m.predict_proba(X_train.iloc[cv_idx])[:, 1])
        cv_scores.append(score)
        print(f'  Fold {fold+1}: {score:.4f}')
    model = RandomForestClassifier(**params)
    model.fit(X_train_us, y_train_us)
    train_auc = roc_auc_score(y_train_us, model.predict_proba(X_train_us)[:, 1])
    val_auc   = roc_auc_score(val_df['isFraud'], model.predict_proba(X_val)[:, 1])
    overfit_gap = train_auc - val_auc
    mlflow.log_params(params)
    mlflow.log_metric('train_auc',   train_auc)
    mlflow.log_metric('val_auc',     val_auc)
    mlflow.log_metric('cv_mean_auc', np.mean(cv_scores))
    mlflow.log_metric('cv_std_auc',  np.std(cv_scores))
    mlflow.log_metric('overfit_gap', overfit_gap)
    print(f'BASELINE    — train: {train_auc:.4f}, val: {val_auc:.4f}, gap: {overfit_gap}')
    print(f'CV          — mean: {np.mean(cv_scores):.4f}, std: {np.std(cv_scores):.4f}')

  Fold 1: 0.8925
  Fold 2: 0.9046
  Fold 3: 0.9098
  Fold 4: 0.8924
  Fold 5: 0.9145
BASELINE    — train: 0.9636, val: 0.8966, gap: 0.06706684909619987
CV          — mean: 0.9028, std: 0.0090
🏃 View run RF_Baseline_Undersampled at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2/runs/705b69b49d7f4b38ba6ed4c52cb84882
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2


In [25]:
# ── RUN 4: MORE TREES ─────────────────────────────────
# RF performance improves with more trees up to a point
# Testing whether 500 trees significantly beats 200
with mlflow.start_run(run_name='RF_MoreTrees_us'):
    params = {'n_estimators': 500, 'max_depth': 15, 'min_samples_leaf': 10,
              'max_features': 'sqrt', 'class_weight': None,
              'random_state': 42, 'n_jobs': -1}
    model = RandomForestClassifier(**params)
    model.fit(X_train_us, y_train_us)
    train_auc = roc_auc_score(y_train_us, model.predict_proba(X_train_us)[:, 1])
    val_auc   = roc_auc_score(val_df['isFraud'], model.predict_proba(X_val)[:, 1])
    overfit_gap = train_auc - val_auc
    mlflow.log_params(params)
    mlflow.log_metric('train_auc',   train_auc)
    mlflow.log_metric('val_auc',     val_auc)
    mlflow.log_metric('overfit_gap', overfit_gap)
    print(f'MORE TREES  — train: {train_auc:.4f}, val: {val_auc:.4f}, gap: {overfit_gap}')

MORE TREES  — train: 0.9644, val: 0.8966, gap: 0.06778876488964924
🏃 View run RF_MoreTrees_us at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2/runs/74a9276938424b78a26e23f35d90098d
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2


In [27]:
with mlflow.start_run(run_name='RF_Tuned_us') as run:
    params = {'n_estimators': 500, 'max_depth': 20, 'min_samples_leaf': 20,
              'max_features': 'sqrt', 'min_samples_split': 10,
              'class_weight': None, 'random_state': 42, 'n_jobs': -1}
    model_best = RandomForestClassifier(**params)
    model_best.fit(X_train_us, y_train_us)
    train_auc = roc_auc_score(y_train_us, model_best.predict_proba(X_train_us)[:, 1])
    val_auc   = roc_auc_score(val_df['isFraud'], model_best.predict_proba(X_val)[:, 1])
    overfit_gap = train_auc - val_auc
    mlflow.log_params(params)
    mlflow.log_metric('train_auc',   train_auc)
    mlflow.log_metric('val_auc',     val_auc)
    mlflow.log_metric('overfit_gap', overfit_gap)
    best_rf_run_id = run.info.run_id
    print(f'TUNED       — train: {train_auc:.4f}, val: {val_auc:.4f}, gap: {overfit_gap}')


TUNED       — train: 0.9715, val: 0.8989, gap: 0.07262505456053792
🏃 View run RF_Tuned_us at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2/runs/b970d4d6d5024f67a5dd6173a55efb93
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2


In [17]:
with mlflow.start_run(run_name='RF_MaxFeatures_30pct_Full_Data'):
    params = {'n_estimators': 500, 'max_depth': 20, 'min_samples_leaf': 20,
              'max_features': 0.3,   # ← try this vs sqrt
              'min_samples_split': 10,
              'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1}
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(val_df['isFraud'], model.predict_proba(X_val)[:, 1])
    overfit_gap = train_auc - val_auc
    mlflow.log_params(params)
    mlflow.log_metric('train_auc',   train_auc)
    mlflow.log_metric('val_auc',     val_auc)
    mlflow.log_metric('overfit_gap', overfit_gap)
    print(f'MAX_FEAT 30% — train: {train_auc:.4f}, val: {val_auc:.4f}, gap: {overfit_gap}')


MAX_FEAT 30% — train: 0.9956, val: 0.9127, gap: 0.08289107632971349
🏃 View run RF_MaxFeatures_30pct_Full_Data at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2/runs/4409cb674a484fd28131784a9928413a
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2


In [30]:
with mlflow.start_run(run_name='RF_MaxFeatures_30pct_Full_Data'):
    params = {'n_estimators': 500, 'max_depth': 20, 'min_samples_leaf': 20,
              'max_features': 0.3,
              'min_samples_split': 10,
              'class_weight': None, 'random_state': 42, 'n_jobs': -1}
    model = RandomForestClassifier(**params)
    model.fit(X_train_us, y_train_us)
    train_auc = roc_auc_score(y_train_us, model.predict_proba(X_train_us)[:, 1])
    val_auc   = roc_auc_score(val_df['isFraud'], model.predict_proba(X_val)[:, 1])
    overfit_gap = train_auc - val_auc
    mlflow.log_params(params)
    mlflow.log_metric('train_auc',   train_auc)
    mlflow.log_metric('val_auc',     val_auc)
    mlflow.log_metric('overfit_gap', overfit_gap)
    print(f'MAX_FEAT 30% — train: {train_auc:.4f}, val: {val_auc:.4f}, gap: {overfit_gap}')

MAX_FEAT 30% — train: 0.9717, val: 0.8976, gap: 0.0740969525923667
🏃 View run RF_MaxFeatures_30pct_Full_Data at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2/runs/2aa1c656d2d3457fb55b9b2a545a7d3f
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2


In [22]:
from sklearn.ensemble import ExtraTreesClassifier

with mlflow.start_run(run_name='RF_ExtraTrees'):
    params = {'n_estimators': 500, 'max_depth': 20,
              'min_samples_leaf': 20, 'max_features': 0.3,
              'min_samples_split': 10, 'class_weight': 'balanced',
              'random_state': 42, 'n_jobs': -1}
    model = ExtraTreesClassifier(**params)
    model.fit(X_train, y_train)
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])
    mlflow.log_params(params)
    mlflow.log_param('estimator_type', 'ExtraTreesClassifier')
    mlflow.log_metric('train_auc',   train_auc)
    mlflow.log_metric('val_auc',     val_auc)
    mlflow.log_metric('overfit_gap', train_auc - val_auc)
    print(f'EXTRA TREES — train: {train_auc:.4f}, val: {val_auc:.4f}')

EXTRA TREES — train: 0.9945, val: 0.9027
🏃 View run RF_ExtraTrees at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2/runs/f48f355337394d83afddca5c921f5a3c
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2


In [18]:
with mlflow.start_run(run_name='RF_MaxFeatures_log2'):
    params = {'n_estimators': 500, 'max_depth': 25,
              'min_samples_leaf': 10, 'max_features': 'log2',
              'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1}
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])
    mlflow.log_params(params)
    mlflow.log_metric('train_auc',   train_auc)
    mlflow.log_metric('val_auc',     val_auc)
    mlflow.log_metric('overfit_gap', train_auc - val_auc)
    print(f'LOG2 FEATURES — train: {train_auc:.4f}, val: {val_auc:.4f}')


LOG2 FEATURES — train: 0.9981, val: 0.9115
🏃 View run RF_MaxFeatures_log2 at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2/runs/4f912b26e0ac4780be7045e202730b53
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2


In [26]:
with mlflow.start_run(run_name='RF_MaxFeatures_log2_us'):
    params = {'n_estimators': 500, 'max_depth': 25,
              'min_samples_leaf': 10, 'max_features': 'log2',
              'class_weight': None, 'random_state': 42, 'n_jobs': -1}
    model = RandomForestClassifier(**params)
    model.fit(X_train_us, y_train_us)
    train_auc = roc_auc_score(y_train_us, model.predict_proba(X_train_us)[:, 1])
    val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])
    mlflow.log_params(params)
    mlflow.log_metric('train_auc',   train_auc)
    mlflow.log_metric('val_auc',     val_auc)
    mlflow.log_metric('overfit_gap', train_auc - val_auc)
    print(f'LOG2 FEATURES — train: {train_auc:.4f}, val: {val_auc:.4f}')

LOG2 FEATURES — train: 0.9858, val: 0.9021
🏃 View run RF_MaxFeatures_log2_us at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2/runs/1e7328fd02d7499282cec7b54fae9584
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2


In [13]:
class TargetDropper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.feature_columns_ = [c for c in X.columns
                                  if c not in ['isFraud', 'TransactionID']]
        return self
    def transform(self, X):
        df = X.drop(columns=['isFraud', 'TransactionID'], errors='ignore').copy()
        if hasattr(self, 'feature_columns_'):
            for col in self.feature_columns_:
                if col not in df.columns:
                    df[col] = -999
            df = df[self.feature_columns_]
        return df

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

mlflow.set_experiment('RandomForest_Training')

with mlflow.start_run(run_name='RF_BestModel_Pipeline') as run:

    print('Step 1: Fitting preprocessors on full 590k rows...')

    cleaner_f  = Cleaner(null_threshold=0.9, fill_value=-999)
    encoder_f  = CategoricalEncoder()
    engineer_f = FeatureEngineer()
    dropper_f  = TargetDropper()
    selector_f = FeatureSelector(variance_threshold=0.01)

    _c    = cleaner_f.fit_transform(train, train['isFraud'])
    _e    = encoder_f.fit_transform(_c)
    _fe   = engineer_f.fit_transform(_e, train['isFraud'])
    dropper_f.fit(_fe)
    _fe_x = dropper_f.transform(_fe)
    y_full = train['isFraud'].reset_index(drop=True)
    X_full = selector_f.fit_transform(_fe_x, y_full).reset_index(drop=True)
    print(f'Processed shape: {X_full.shape}')

    # ── Train final model on full data ────────────────────
    # Best params: RF_MaxFeatures_30pct_Full_Data
    # full data + class_weight=balanced beats undersampling for RF
    print('Step 2: Training final model...')
    best_params = {
        'n_estimators':    500,
        'max_depth':       20,
        'min_samples_leaf': 20,
        'max_features':    0.3,
        'min_samples_split': 10,
        'class_weight':    'balanced',
        'random_state':    42,
        'n_jobs':          -1
    }
    final_rf = RandomForestClassifier(**best_params)
    final_rf.fit(X_full, y_full)
    train_auc = roc_auc_score(y_full, final_rf.predict_proba(X_full)[:, 1])
    print(f'Train AUC: {train_auc:.4f}')

    # ── Build full pipeline ───────────────────────────────
    print('Step 3: Building pipeline...')
    full_pipeline = Pipeline([
        ('cleaner',  cleaner_f),
        ('encoder',  encoder_f),
        ('engineer', engineer_f),
        ('dropper',  dropper_f),
        ('selector', selector_f),
        ('model',    final_rf),
    ])

    # ── Verify on raw sample ─────────────────────────────
    print('Step 4: Verifying on raw sample...')
    raw_sample = train.iloc[:100].drop(columns=['isFraud'])
    test_preds = full_pipeline.predict_proba(raw_sample)[:, 1]
    print(f'Sample preds: min={test_preds.min():.3f}, max={test_preds.max():.3f}')
    print('Pipeline verified!')

    # ── Log and save ──────────────────────────────────────
    mlflow.log_params(best_params)
    mlflow.log_param('pipeline_steps',
                     'Cleaner→CategoricalEncoder→FeatureEngineer→TargetDropper→FeatureSelector→RF')
    mlflow.log_param('trained_on',      'full_590k_rows')
    mlflow.log_param('balancing',       'class_weight=balanced_no_undersampling')
    mlflow.log_param('why_no_us',       'full_data+balanced_beats_undersampling_for_RF')
    mlflow.log_param('can_run_on_raw',  True)
    mlflow.log_metric('train_auc',      train_auc)
    mlflow.log_metric('val_auc',        0.9127)
    mlflow.log_metric('overfit_gap',    0.0829)
    mlflow.log_metric('n_features',     X_full.shape[1])

    mlflow.sklearn.log_model(full_pipeline, 'full_rf_pipeline')
    rf_run_id = run.info.run_id
    print(f'\nFull RF pipeline saved!')
    print(f'Run ID: {rf_run_id}')

print(f'\nRF val_auc: 0.9127 vs XGBoost val_auc: 0.9271')
print('XGBoost remains BestFraudDetector in Model Registry')
print('RF pipeline saved for comparison and inference reference')

Step 1: Fitting preprocessors on full 590k rows...
clean_in
🏃 View run RandomForest_Cleaning at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2/runs/fc6b03cd7ddb4885a7cc8d46881baa16
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2
[Cleaner] Done — dropped 12 cols, 590540 rows
clean trans in
encode_in
🏃 View run RandomForest_Categorical_Encoding at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2/runs/0d39de8432e94805a53774ae7c184032
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2
[CategoricalEncoder] Done — OHE cols: 13, LE cols: 6
encode trans in
🏃 View run RF_Feature_Engineering at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2/runs/0edec294967e4cc9a680e88c7cd54cd0
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2
select_in
🏃 View run RandomForest_Feature_Selection at: https://dagshub.com/gvakh23/ML_assignment2.m

2026/05/06 08:35:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 08:35:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Full RF pipeline saved!
Run ID: 6a7a00873a184a9ba6ee6db12a38fdcb
🏃 View run RF_BestModel_Pipeline at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2/runs/6a7a00873a184a9ba6ee6db12a38fdcb
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/2

RF val_auc: 0.9127 vs XGBoost val_auc: 0.9271
XGBoost remains BestFraudDetector in Model Registry
RF pipeline saved for comparison and inference reference
